# Exploration for Q6: a measure of need that is not demand
Owner: Valentino. Read `questions/q6_divergence_measure/README.md` and `addendum.md` first.

Order of work, and the rule: **nothing in step 0 is written after a result is seen.**

0. Definition of need, inputs with their defence, rejection and revision criteria
1. What Q1 already established, and the Wilkin test case
2. Build the index (not before step 0 is filled in)
3. Divergence against each of the three metrics
4. Stability under other weights
5. Map, and county characteristics that explain the divergence

## Step 0: written before any data

**Definition of need (one sentence that could turn out to be false):**

> A household is in need if, at any point in the past year, it worried about running out of
> food, regardless of whether it sought help.

This is the first item of the USDA food-security questionnaire, so a survey can measure it
directly and disagree with us. "Regardless of whether it sought help" is what separates
need from demand: the worry exists whether or not the household asks anyone. A county's
need is the number (or share) of its households in that state.

**Inputs, argued one by one.** An input belongs only if all three answers hold up, and
the mechanism has to end in "...so the household is more likely to worry about running
out of food". If it ends in "...so the household is less likely to visit a food shelf",
it is a barrier, not need, and it goes to step 5.

| input | measurable at zero demand? | in MMG? | mechanism | have it? |
|---|---|---|---|---|
| `pct_hh_snap` | yes | no | a household on SNAP has already had to ask for help with food; benefits run out before the month does, so it worries about running out of food | yes (ACS B22003) |
| `pct_no_vehicle` | yes | no | without a vehicle there is no way to carry a week's groceries home, so the household worries about running out of food. (It is also a barrier to reaching a food shelf; it is used on the need side only, never in step 5) | yes (ACS B08201) |
| `senior_poverty_rate` | yes | no (MMG uses all-ages poverty) | poor seniors live on a fixed income with no savings and no future earnings, so a bill they cannot defer becomes a month they worry about food | yes (ACS B17001) |
| housing-cost burden | yes | no | rent or mortgage is paid first; a household paying over 30% of income on housing has little left, so any other bill means worrying about food | no: ACS B25070 (renters) and B25091 (owners), one more pull |
| low access to a grocery store (LILA share) | yes | no | low income and no store within reach: the household worries first about how to get food at all, then how to afford it | partly: USDA Atlas is in the repo (Q8), needs a county aggregate |
| educational attainment (share of adults without a high-school diploma) | yes | no | low education closes doors when trouble comes, so a household that gets into trouble has fewer ways out and worries about food | no: ACS B15003, one more pull |
| credit delinquency | yes | no | a household unable to pay back what it borrowed has to keep finding new financing, so it worries about affording food. Doubt: delinquents were once approved for credit, so they had some standing; the poorest households often have no credit file at all | no: Urban Institute county data, new source. Last priority, because of the doubt |

**Barriers, kept for step 5 (why need does not become a visit):** `mean_km_to_site`,
`rucc_2023`, `pct_senior` (mobility, reluctance to ask). Not inputs to need.

**Left out:** `poverty_rate`, `median_hh_income`, `unemployment_rate` (MMG inputs);
everything derived from activity (demand).

**Evidence that would make us reject the definition:**

> Survey the counties we rank highest on the measure. If the share of households that
> worried about running out of food is no higher than in the rest of the state, the
> measure is wrong.

**Evidence that would make us revise the measure (definition survives, implementation does not):**

> 1. The index is poverty in a costume: regressed on the poverty rate it leaves nothing
>    behind (R2 near 0.9 or above, or the top ten counties identical to the poverty top ten).
>    Need should correlate with poverty; it should not *be* poverty.
> 2. An input's mechanism fails on inspection, or the input turns out to track activity
>    (downstream of visits). Drop it and rebuild.
> 3. The top ten changes under reasonable alternative weights. Then the ranking is not a
>    finding.

**Prediction for Wilkin:** the measure must give Wilkin a need estimate although it has
no recorded activity. The definition allows it: Wilkin has need as long as people live
there.

> Written 1 October, before the index exists: Wilkin ranks around the 70th percentile of
> need (roughly 25th of 87 from the top), driven by SNAP households and no-vehicle
> households, both above the state median. On MMG's own inputs it is below median
> (poverty rate rank 68), so the measure and MMG should disagree on this county.

## Step 1: what Q1 established
- Recorded activity rises faster than need (log-log slope 1.15 to 1.24).
- The Q1 residual is demand relative to MMG need: positive = more recorded than need predicts.
- The residual partly measures the network: counties with more sites record more activity
  per food-insecure person. Pounds is a supply measure as much as a demand measure.
- Distance, RUCC group, no-vehicle households and SNAP households explain 3% of the residual.
- Moran's I = 0.06: no general spatial pattern; border pairs are examples, not evidence.
- Wilkin has food-insecure people and no reporting food shelf.

So the Q1 residual is not a need measure, and the Q6 measure must not be built from it.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "qkit.py").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

import config as cfg
import qkit

In [ ]:
# County-year frame, as in Q1: activity + MMG need + public covariates, 2022-2024
df = qkit.county_year(cfg.Q1_YEARS).merge(
    qkit.covariates(cfg.Q1_YEARS).drop(columns="county_name"),
    on=["fips", "year"], how="left", validate="1:1")
no_activity = df["households"].isna() & df["fi_persons"].notna()
for m in cfg.THREE_METRICS:
    df.loc[no_activity, m] = 0
df.loc[no_activity, "months_observed"] = 0

# The Q1 residual ranking, from the script that rebuilds it
q1 = pd.read_csv(cfg.TAB / "q1_residual_ranking.csv", dtype={"fips": str})

print(df.shape, "|", df["fips"].nunique(), "counties |", sorted(df["year"].unique()))
print("columns available as inputs:")
print(sorted(c for c in df.columns if c not in cfg.THREE_METRICS))

In [ ]:
# The test case: what do we know about Wilkin without any activity data?
cols = ["year", "fi_persons", "fi_rate", "population", "poverty_rate", "pct_no_vehicle",
        "pct_hh_snap", "pct_senior", "rucc_2023", "n_sites_reporting", "mean_km_to_site"]
df[df["county_name"] == "Wilkin"][cols]

## Step 2: the index
Decisions, recorded before any ranking was seen (1 October):

| decision | choice | why |
|---|---|---|
| inputs | the seven in step 0: SNAP, no vehicle, senior poverty, housing burden, no HS diploma, LILA share, debt in collections | each argued above |
| LILA definition | `lila_share_1and10`, USDA's headline | easiest to defend; the vehicle version double-counts `pct_no_vehicle` and is kept for the stability check |
| scale | z-score of each input within each year | puts shares of different sizes on one footing; a coefficient of 1 = one standard deviation |
| weights | equal, 1/7 each | no argument was found for more; alternatives come in step 4 as a stability check, not before |
| direction | higher = more need, for every input | |
| county score | mean of the three years | the ranking is a 2022-2024 statement, as in Q1 |

The index is relative: it ranks counties, it does not say what share of households worried.
That share is what the definition is about and only a survey measures it, which is why the
rejection criterion is a survey.

**Checks that follow from the revision criteria** (results in the next cell):
1. Is it poverty in a costume? Regress the index on the poverty rate; R2 near 0.9 or above
   rejects it. Compare the top ten with the poverty top ten.
2. Does one input dominate? Correlations among inputs, and the first principal component's
   share and loadings. One input carrying most of the variance means the index is that input.
3. Wilkin against the prediction in step 0.

In [ ]:
INPUTS = ["pct_hh_snap", "pct_no_vehicle", "senior_poverty_rate", "pct_hh_housing_burden",
          "pct_no_hs_diploma", "lila_share_1and10", "pct_debt_collections"]
WEIGHTS = pd.Series(1 / len(INPUTS), index=INPUTS)

def build_index(frame, inputs=INPUTS, weights=WEIGHTS):
    """Z-score each input within year, weighted mean -> need index per county-year."""
    z = frame.groupby("year")[inputs].transform(lambda s: (s - s.mean()) / s.std())
    return (z * weights).sum(axis=1) / weights.sum()

df["need_index"] = build_index(df)
need = (df.groupby(["fips", "county_name"], as_index=False)
          .agg(need_index=("need_index", "mean"), poverty_rate=("poverty_rate", "mean"),
               fi_rate=("fi_rate", "mean"), population=("population", "mean")))
need["need_rank"] = need["need_index"].rank(ascending=False).astype(int)
need = need.sort_values("need_rank").reset_index(drop=True)

w = need[need["county_name"] == "Wilkin"].iloc[0]
print(f"Wilkin: index {w['need_index']:.2f}, rank {w['need_rank']} of 87 "
      f"(percentile {100 * (1 - (w['need_rank'] - 0.5) / 87):.0f}). Prediction: ~70th percentile, ~25th.")
print("top 10 need:   ", ", ".join(need["county_name"].head(10)))
print("bottom 10 need:", ", ".join(need["county_name"].tail(10)[::-1]))
need[["need_rank", "county_name", "need_index", "poverty_rate", "fi_rate"]].head(15).round(2)

In [ ]:
# Check 1: poverty in a costume?
fit = smf.ols("need_index ~ poverty_rate", data=need).fit()
pov_top = set(need.nlargest(10, "poverty_rate")["county_name"])
mmg_top = set(need.nlargest(10, "fi_rate")["county_name"])
top = set(need.head(10)["county_name"])
print(f"index on poverty rate: R2 = {fit.rsquared:.2f} (revise if ~0.9 or above)")
print(f"top ten shared with poverty top ten: {len(top & pov_top)} | with MMG fi_rate top ten: {len(top & mmg_top)}")
print("Spearman with poverty rate:", round(need["need_index"].corr(need["poverty_rate"], method="spearman"), 2),
      "| with MMG fi_rate:", round(need["need_index"].corr(need["fi_rate"], method="spearman"), 2))

# Check 2: does one input dominate?
z = df.groupby("year")[INPUTS].transform(lambda s: (s - s.mean()) / s.std())
print("\ncorrelations among inputs (county-years):")
display(z.corr().round(2))
U, S, Vt = np.linalg.svd(z.values - z.values.mean(axis=0), full_matrices=False)
print("share of input variance on the first component:", round(S[0] ** 2 / (S ** 2).sum(), 2))
print("loadings on it:", dict(zip(INPUTS, Vt[0].round(2))))
print("correlation of each input with the index:")
print(z.corrwith(df["need_index"]).round(2).to_string())

## Step 3: where need and recorded activity diverge
Divergence is computed for each of the three metrics, as the brief asks. Two views:

1. **Rate against rate.** Activity per 1,000 residents (mean of 2022-2024), ranked, against
   the need rank. `divergence = activity_rank - need_rank`, so **positive = the county
   ranks higher on need than on recorded activity**: need that is not showing up as demand.
   Wilkin has zero activity and ranks last on every metric.
2. **Against Q1's residual.** The Q1 residual is activity relative to MMG need. If the new
   measure captures need MMG misses, counties high on the index should sit below the Q1
   line. A county high on need and negative on the residual is divergent on both views.

**Prediction, written before running (1 October):** Wilkin, Nobles and Grant are in the
top of the divergence list. Nobles and Grant were among the Q1 counties recording less
than need predicts, and Nobles is 3rd on the index.

In [ ]:
act = (df.groupby(["fips", "county_name"])
         .apply(lambda g: pd.Series({m + "_per_1k": 1000 * g[m].sum() / g["population"].sum()
                                     for m in cfg.THREE_METRICS}), include_groups=False)
         .reset_index())
div = need.merge(act, on=["fips", "county_name"])
for m in cfg.THREE_METRICS:
    div[m + "_rank"] = div[m + "_per_1k"].rank(ascending=False).astype(int)
    div["div_" + m] = div[m + "_rank"] - div["need_rank"]
div["div_mean"] = div[["div_" + m for m in cfg.THREE_METRICS]].mean(axis=1)
div = div.sort_values("div_mean", ascending=False).reset_index(drop=True)

print("Spearman, need index vs activity per 1,000 residents:",
      {m: round(div["need_index"].corr(div[m + "_per_1k"], method="spearman"), 2) for m in cfg.THREE_METRICS})
pred = ["Wilkin", "Nobles", "Grant"]
print("predicted counties, position in the divergence list:",
      {c: int(div.index[div["county_name"] == c][0]) + 1 for c in pred})
print("\nneed ranks higher than recorded activity (top 12):")
cols = ["county_name", "need_rank"] + [m + "_rank" for m in cfg.THREE_METRICS] + ["div_mean"]
display(div[cols].head(12).round(1))
print("recorded activity ranks higher than need (top 8):")
div[cols].tail(8)[::-1].round(1)

In [ ]:
# View 1 as a chart: need rank against activity rank, one panel per metric.
# On the diagonal, need and recorded activity agree. Below it: need outranks activity.
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5), sharey=True)
for ax, m in zip(axes, cfg.THREE_METRICS):
    ax.scatter(div["need_rank"], div[m + "_rank"], alpha=0.6)
    ax.plot([1, 87], [1, 87], color="grey", ls="--", lw=0.8, label="need and activity agree")
    d = div[m + "_rank"] - div["need_rank"]
    for idx in pd.concat([d.nlargest(6), d.nsmallest(4)]).index:
        ax.annotate(div.loc[idx, "county_name"], (div.loc[idx, "need_rank"], div.loc[idx, m + "_rank"]),
                    xytext=(4, 4), textcoords="offset points", fontsize=8)
    ax.invert_xaxis(); ax.invert_yaxis()          # rank 1 at the top right, like a league table
    ax.set_xlabel("need rank (1 = most need)")
    ax.set_title(f"{m} per 1,000 residents")
axes[0].set_ylabel("activity rank (1 = most recorded)")
axes[0].legend(fontsize=8, loc="lower left")
fig.suptitle("Where need outranks recorded activity (below the line) and the reverse (above)")
plt.tight_layout()
plt.show()

In [ ]:
# View 2: the index against Q1's residual (activity relative to MMG need)
both = need.merge(q1[["fips", "mean_resid"]], on="fips", how="left")
print("Spearman, need index vs Q1 residual:", round(both["need_index"].corr(both["mean_resid"], method="spearman"), 2),
      "(Wilkin has no residual)")
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(both["need_index"], both["mean_resid"], c=both["need_rank"], cmap="viridis_r",
                vmin=1, vmax=87, alpha=0.85, edgecolor="white", lw=0.5)
cb = plt.colorbar(sc, ax=ax); cb.set_label("need rank (1 = most need)"); cb.ax.invert_yaxis()
ax.axhline(0, color="grey", lw=0.8)
ax.axvline(0, color="grey", lw=0.8)
both = both.merge(div[["fips", "div_mean"]], on="fips")
show = pd.concat([both.nlargest(8, "div_mean"), both.nsmallest(4, "div_mean"), both.nlargest(3, "need_index")])
for _, r in show.drop_duplicates("fips").iterrows():
    ax.annotate(r["county_name"], (r["need_index"], r["mean_resid"]), xytext=(4, 4),
                textcoords="offset points", fontsize=8)
ax.set_xlabel("need index (higher = more need)")
ax.set_ylabel("Q1 residual: log activity relative to MMG need")
ax.set_title("View 2: the index against what Q1 left unexplained. Labels: most divergent counties")
plt.tight_layout()
plt.show()
print("high need, below the Q1 line (need rank <= 20 and residual < 0):")
both[(both["need_rank"] <= 20) & (both["mean_resid"] < 0)].sort_values("need_rank")[["need_rank", "county_name", "need_index", "mean_resid"]].round(2)

## Step 4: stability
The README's rule: show stability or do not show a ranking. Each variant is compared with
the baseline on three things: how many of the baseline top ten it keeps, the rank
correlation of the full ordering, and where Wilkin lands.

| variant | what it tests |
|---|---|
| drop SNAP and debt | the two inputs closest to poverty (step 2 check) |
| LILA vehicle definition | the other USDA definition |
| rank scaling | whether z-scores let a few extreme counties drive the index |
| leave one input out, seven times | whether any single input carries the ranking |
| 500 random weight draws | how often each county is in the top ten under any weights |

**Prediction, written before running:** *fill in: which variant moves the top ten most?*

In [ ]:
def rank_index(frame, inputs):
    r = frame.groupby("year")[inputs].rank(pct=True)
    return r.mean(axis=1)

def county_scores(score):
    s = df.assign(score=score).groupby("county_name")["score"].mean()
    return s.rank(ascending=False).astype(int)

base = county_scores(df["need_index"])
base_top = set(base[base <= 10].index)

variants = {
    "drop SNAP and debt": county_scores(build_index(df, [i for i in INPUTS if i not in ("pct_hh_snap", "pct_debt_collections")])),
    "LILA vehicle": county_scores(build_index(df, [i if i != "lila_share_1and10" else "lila_share_vehicle" for i in INPUTS])),
    "rank scaling": county_scores(rank_index(df, INPUTS)),
}
for i in INPUTS:
    variants["without " + i] = county_scores(build_index(df, [j for j in INPUTS if j != i]))

rows = []
for name, r in variants.items():
    rows.append({"variant": name, "top10 kept": len(base_top & set(r[r <= 10].index)),
                 "spearman with baseline": round(base.corr(r, method="spearman"), 3),
                 "Wilkin rank": int(r["Wilkin"]),
                 "enters top 10": ", ".join(sorted(set(r[r <= 10].index) - base_top))})
stab = pd.DataFrame(rows)
print("baseline top ten:", ", ".join(base[base <= 10].sort_values().index))
display(stab)

# Random weights: 500 draws from a flat Dirichlet, i.e. any weights that sum to one
rng = np.random.default_rng(0)
hits = pd.Series(0, index=base.index)
wilkin = []
for _ in range(500):
    w = pd.Series(rng.dirichlet(np.ones(len(INPUTS))), index=INPUTS)
    r = county_scores(build_index(df, INPUTS, w))
    hits[r[r <= 10].index] += 1
    wilkin.append(int(r["Wilkin"]))
share = (hits / 500).sort_values(ascending=False)
print("\nshare of 500 random weightings in which each county is in the top ten:")
print(share.head(15).round(2).to_string())
print(f"Wilkin rank across draws: median {int(np.median(wilkin))}, range {min(wilkin)} to {max(wilkin)}")